# Problem 15 - Traveling Salesperson Problem (TSP)

## Stage 5 - 2-Opt Local Search + Simulated Annealing

In Stages 1-4 every algorithm **builds** a route from scratch:

- Brute Force and Held-Karp are exact, but too slow for many cities.
- Nearest Neighbor and the MST approximation are fast, but their routes are often 10-30 % too long and contain **crossing edges**.

Stage 5 works differently: it takes a route that already exists (for example from Nearest Neighbor) and **improves it step by step** with small changes. This is called **local search**.

## Basic Ideas

- **Crossing edges are always bad.** If two edges of a route cross each other, we can "untangle" them and the route gets shorter (triangle inequality).
- **2-Opt move:** remove 2 edges of the route and reconnect the two pieces the other way. In the list this is just **reversing a part of the route**.
- **Local minimum:** a route where no single 2-Opt move helps anymore. It is good, but not always optimal.
- **Simulated Annealing:** sometimes accept a move that makes the route a bit **worse**, so that we can escape from a local minimum.

## Distance Matrix

We use the same 4-city matrix as in the previous stages.

In [1]:
D = [
    [0, 10, 15, 20],
    [10, 0, 35, 25],
    [15, 35, 0, 30],
    [20, 25, 30, 0]
]

print("Number of cities:", len(D))

Number of cities: 4


## Calculating the Total Route Distance

Same helper function as in Stage 1. A route starts and ends in the same city, for example `[3, 0, 1, 2, 3]`.

In [2]:
def route_cost(route, D):
    total = 0

    for i in range(len(route) - 1):
        from_city = route[i]
        to_city = route[i + 1]

        total += D[from_city][to_city]

    return total

## Step 1 - One 2-Opt Move

We take two edges of the route that do **not** touch each other:

- edge 1: `(u1, v1) = (route[i], route[i + 1])`
- edge 2: `(u2, v2) = (route[j], route[j + 1])`

We **reverse the part `route[i+1 .. j]`**. After that the route uses the edges `(u1, u2)` and `(v1, v2)` instead. All other edges stay the same (the reversed part is driven backwards, and the matrix is symmetric).

Because only 4 edges change, we do not need to add up the whole route again. The change of the length is:

**ΔL = D[u1][u2] + D[v1][v2] − D[u1][v1] − D[u2][v2]**

This takes **O(1)** time, no matter how many cities there are.

- ΔL < 0: the route gets shorter, the move is good.
- ΔL > 0: the route gets longer.

In [3]:
def two_opt_delta(route, i, j, D):
    u1, v1 = route[i], route[i + 1]      # first edge we remove
    u2, v2 = route[j], route[j + 1]      # second edge we remove

    added = D[u1][u2] + D[v1][v2]        # two new edges
    removed = D[u1][v1] + D[u2][v2]      # two old edges

    return added - removed


def two_opt_move(route, i, j):
    # Reverse the part route[i+1 .. j]
    return route[:i + 1] + route[i + 1:j + 1][::-1] + route[j + 1:]

## Hand Trace from the Exam (Question 3)

Greedy from city 3 gave the route **3 → 0 → 1 → 2 → 3** with distance **95**.

We remove the edges **(3, 0)** and **(1, 2)** and add **(3, 1)** and **(0, 2)**:

- edge (3, 0) = `route[0] → route[1]`, so **i = 0**
- edge (1, 2) = `route[2] → route[3]`, so **j = 2**

In [4]:
greedy_route = [3, 0, 1, 2, 3]
i, j = 0, 2

u1, v1 = greedy_route[i], greedy_route[i + 1]
u2, v2 = greedy_route[j], greedy_route[j + 1]
delta = two_opt_delta(greedy_route, i, j, D)

print("Before:", greedy_route, "Distance:", route_cost(greedy_route, D))
print(f"u1={u1}, v1={v1}, u2={u2}, v2={v2}")
print(f"Delta L = D[{u1}][{u2}] + D[{v1}][{v2}] - D[{u1}][{v1}] - D[{u2}][{v2}]")
print(f"        = {D[u1][u2]} + {D[v1][v2]} - {D[u1][v1]} - {D[u2][v2]} = {delta}")

new_route = two_opt_move(greedy_route, i, j)
print("\nAfter: ", new_route, "Distance:", route_cost(new_route, D))
print("Check: 95 + Delta L =", 95 + delta)

Before: [3, 0, 1, 2, 3] Distance: 95
u1=3, v1=0, u2=1, v2=2
Delta L = D[3][1] + D[0][2] - D[3][0] - D[1][2]
        = 25 + 15 - 20 - 35 = -15

After:  [3, 1, 0, 2, 3] Distance: 80
Check: 95 + Delta L = 80


One single 2-Opt move removed the expensive edge 1 → 2 (35) and turned the Greedy route (95) into the **optimal route (80)**.

## Step 2 - Full 2-Opt Algorithm

One move is not enough for big maps. We repeat:

1. Try every pair of edges (i, j).
2. If a move makes the route shorter (ΔL < 0), do it.
3. Stop when a full pass finds no improvement. The route is now **2-optimal** (a local minimum).

**Complexity:** one pass checks about n²/2 pairs, each in O(1), so **O(n²) per pass**. The number of passes is usually small, but there is no good upper bound in the worst case.

In [5]:
def two_opt(route, D, show_steps=False):
    route = route[:]
    n = len(route) - 1                       # number of cities (start city is repeated at the end)
    improved = True

    while improved:
        improved = False
        for i in range(n - 1):
            for j in range(i + 2, n):
                if i == 0 and j == n - 1:
                    continue                 # these two edges touch city route[0]
                delta = two_opt_delta(route, i, j, D)
                if delta < -1e-9:
                    route = two_opt_move(route, i, j)
                    improved = True
                    if show_steps:
                        print(f"i={i}, j={j}, delta={delta:.1f} -> {route}, distance {route_cost(route, D):.1f}")

    return route, route_cost(route, D)


route, distance = two_opt([3, 0, 1, 2, 3], D, show_steps=True)
print("\n2-Opt route:", route)
print("Total distance:", distance)

i=0, j=2, delta=-15.0 -> [3, 1, 0, 2, 3], distance 80.0

2-Opt route: [3, 1, 0, 2, 3]
Total distance: 80


## Step 3 - The Problem: Local Minimum

2-Opt only accepts moves that make the route **shorter**. Imagine walking down a mountain in fog with the rule "always step down": you will reach the bottom of *some* valley, but maybe not the deepest one. From there every step goes up, so you are stuck.

## Step 4 - Simulated Annealing

The name comes from metal work: hot metal is cooled **slowly**, so the atoms can move around and settle in a very good structure.

- We have a **temperature T**.
- A good move (ΔL < 0) is **always** accepted.
- A bad move (ΔL > 0) is accepted with probability **P = e^(−ΔL / T)**.
- After each round we cool down: **T ← γ · T** with γ = 0.995.

When T is high, many bad moves are accepted (explore, escape from valleys). When T is low, almost only good moves are accepted, so it behaves like normal 2-Opt.

In [6]:
import math

delta_bad = 10
print("Probability to accept a move that is 10 units worse:")
for T in [100, 10, 1]:
    print(f"  T = {T:>3}:  P = e^(-10/{T}) = {math.exp(-delta_bad / T):.5f}")

Probability to accept a move that is 10 units worse:
  T = 100:  P = e^(-10/100) = 0.90484
  T =  10:  P = e^(-10/10) = 0.36788
  T =   1:  P = e^(-10/1) = 0.00005


In [7]:
import random

def simulated_annealing(route, D, T_start=None, gamma=0.995, T_end=0.001, moves_per_T=None,
                        seed=0, history=False):
    rng = random.Random(seed)
    route = route[:]
    n = len(route) - 1
    current = route_cost(route, D)
    best_route, best = route[:], current

    if n < 4:
        return (best_route, best, []) if history else (best_route, best)
    if T_start is None:                       # 10 % of the average distance between two cities
        T_start = 0.1 * sum(map(sum, D)) / (n * (n - 1))
    if moves_per_T is None:
        moves_per_T = 10 * n

    T, step, log = T_start, 0, []
    while T > T_end:
        for _ in range(moves_per_T):
            i, j = sorted(rng.sample(range(n), 2))         # random pair of edges
            if j - i < 2 or (i == 0 and j == n - 1):
                continue
            delta = two_opt_delta(route, i, j, D)
            if delta < 0 or rng.random() < math.exp(-delta / T):
                route = two_opt_move(route, i, j)
                current += delta
                if current < best:                        # remember the best route ever seen
                    best_route, best = route[:], current
            step += 1
            if history and step % 200 == 0:
                log.append((step, current, best, T))
        T *= gamma                                        # cool down

    best_route, best = two_opt(best_route, D)             # final polish with normal 2-Opt
    return (best_route, best, log) if history else (best_route, best)

## Stage 5 on the Exam Example

**V5a = Greedy + 2-Opt** and **V5b = 2-Opt + Simulated Annealing** both start from a constructed route.
Here we start from the Greedy route of Stage 2 (from city 3, distance 95).

In [8]:
greedy_route = [3, 0, 1, 2, 3]
print("Start (Greedy):     ", greedy_route, "Distance:", route_cost(greedy_route, D))

route_a, cost_a = two_opt(greedy_route, D)
print("V5a Greedy + 2-Opt: ", route_a, "Distance:", cost_a)

route_b, cost_b = simulated_annealing(greedy_route, D, seed=0)
print("V5b 2-Opt + SA:     ", route_b, "Distance:", cost_b)

Start (Greedy):      [3, 0, 1, 2, 3] Distance: 95
V5a Greedy + 2-Opt:  [3, 1, 0, 2, 3] Distance: 80
V5b 2-Opt + SA:      [3, 2, 0, 1, 3] Distance: 80


## Conclusion

- **2-Opt** fixes the main weakness of Greedy and MST: it removes crossing edges. On the exam example one move turns 95 into the optimal 80.
- The change of length of a move is computed in **O(1)**; one full pass of 2-Opt takes **O(n²)**.
- **Simulated Annealing** accepts some worse moves at high temperature and escapes local minima, where pure 2-Opt gets stuck.
- **Advantages:** routes very close to OPT, works for large maps, can start from any route.
- **Disadvantages:** no mathematical guarantee, and SA needs parameters (start temperature, γ) and more time.

The comparison with all other stages on random maps (Part C) is in **`tsp_part_c_experiments.ipynb`**.